# Даты договора ТЭ в озере и даты Снигура

Моя сборка ТЭ (`trade_acquiring_clients_month_excel.ipynb`) и final_df берут из `ods_alpha.scd1_agreements` только `d_valid_from` и `d_valid_to`, из `scd1_agr_terms` - те же две даты привязки `P`. У Снигура три даты: начало, окончание действия и закрытие, плюс статус.

Тетрадка:
1. выводит все колонки трёх таблиц и отбирает колонки-даты: `ods_alpha.scd1_agreements`, `ods_alpha.scd1_agr_terms`, `ods.scd1_z_r2_ip_merchants` (договор-мерчант в ЦФТ, связь `id = abs_agr_id`);
2. по всем договорам Снигура поднимает эти даты (договор - по номеру, мерчант - через `abs_agr_id`, привязки `P` - последняя дата окончания);
3. для каждой даты Снигура ранжирует колонки озера: сколько совпадает точно, сколько с разницей в день, пусто ли там же, где у Снигура. Отдельно - по статусам Снигура;
4. для найденной колонки закрытия считает, насколько правило «действует на дату» по озеру совпадает с действующими у Снигура.

Если какая-то колонка совпадает с «Датой закрытия» Снигура, её можно взять в сборку вместо `d_valid_to` или вместе с ним.


In [ ]:
import getpass
import re
from pathlib import Path

import pandas as pd
from IPython.display import display
from rail_connectors.connection import connect

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 240)
pd.set_option('display.max_rows', 200)

AS_OF_DATE = '2026-09-01'
BASE_DIR = Path('/home/jovyan/documents/Equaring')
SNIGUR_FILE = BASE_DIR / 'Скрипт для Ани по сборке продукта эквайринг' / 'cdwh_acq_clients.xlsx'
SNIGUR_SHEET = 0
OUT_FILE = BASE_DIR / ('te_close_date_sources_%s.xlsx' % AS_OF_DATE)

TABLES = {
    'agr': 'ods_alpha.scd1_agreements',
    'terms': 'ods_alpha.scd1_agr_terms',
    'cft': 'ods.scd1_z_r2_ip_merchants',
}
# Колонки, выбранные вручную. None - взять лучшую по совпадению
START_COL = None
END_COL = None
CLOSE_COL = None

as_of = pd.Timestamp(AS_OF_DATE)
as_of_ru = as_of.strftime('%d.%m.%Y')

LAKE_USER = input('Логин Impala: ').strip()
LAKE_PASSWORD = getpass.getpass('Пароль Impala: ')
imp = connect(
    to='IMPALA',
    extra_options={'db': 'sandbox_ai'},
    driver_args={'tez.queue.name': 'ai'},
    kerberos={
        'keytab_path': '/home/jovyan/test_requests/tech.keytab',
        'use_credentials': True,
        'update_keytab': True,
    },
    user_params={'user_name': LAKE_USER, 'password': LAKE_PASSWORD},
)
imp._init_connection()

def fetch(sql):
    with imp:
        imp.execute('set MEM_LIMIT=16g')
        df = imp.fetch(sql)
    df.columns = [str(c).split('.')[-1].lower() for c in df.columns]
    return df

def sql_in(vals):
    return ', '.join("'%s'" % str(v).replace("'", "''") for v in vals)

def chunks(vals, n=1000):
    vals = sorted(set(v for v in vals if v is not None and not pd.isna(v)))
    return [vals[i:i + n] for i in range(0, len(vals), n)]
print('imp ready')


## 1. Колонки таблиц

In [ ]:
DATE_NAME = re.compile(r'(?:^d_|^dt_|date|^c_dat|_dt$|_date$)', re.I)

desc_parts = []
for alias, table in TABLES.items():
    d = fetch('DESCRIBE %s' % table)
    d = d.rename(columns={d.columns[0]: 'Колонка', d.columns[1]: 'Тип'})
    if len(d.columns) > 2:
        d = d.rename(columns={d.columns[2]: 'Комментарий'})
    else:
        d['Комментарий'] = None
    d['Колонка'] = d['Колонка'].astype(str).str.strip().str.lower()
    d['Тип'] = d['Тип'].astype(str).str.strip().str.lower()
    d.insert(0, 'Таблица', table)
    d.insert(1, 'alias', alias)
    desc_parts.append(d[['Таблица', 'alias', 'Колонка', 'Тип', 'Комментарий']])
desc = pd.concat(desc_parts, ignore_index=True)
comment = desc['Комментарий'].fillna('').astype(str).str.lower()
desc['Похоже на дату'] = (
    desc['Тип'].str.contains('timestamp|date')
    | desc['Колонка'].str.contains(DATE_NAME)
    | comment.str.contains('дата|date')
)
date_desc = desc[desc['Похоже на дату']].reset_index(drop=True)
print('Колонок всего:', desc.groupby('Таблица').size().to_dict())
display(date_desc)


## 2. Договоры Снигура и их даты в озере

In [ ]:
EMPTY = ('', 'NONE', 'NAN', '<NA>', 'NAT', 'NULL')

def _low(v):
    return re.sub(r'\s+', ' ', str(v)).strip().lower()

def norm_agr(v):
    if v is None or pd.isna(v):
        return None
    s = str(v).strip()
    if re.fullmatch(r'\d+\.0+', s):
        s = s.split('.')[0]
    s = re.sub(r'\s+', '', s).upper()
    return None if s in EMPTY else s

def to_date(s):
    s = pd.Series(s, dtype=object)
    iso = s.map(lambda v: isinstance(v, str) and bool(re.match(r'\s*\d{4}-\d{2}-\d{2}', v)))
    out = pd.to_datetime(s.where(~iso), errors='coerce', dayfirst=True)
    out[iso] = pd.to_datetime(s[iso].str.strip().str[:10], errors='coerce', format='%Y-%m-%d')
    out = out.dt.normalize()
    return out.where(out < pd.Timestamp('2100-01-01'))

def read_with_header(path, sheet, marker='договор', max_rows=20):
    raw = pd.read_excel(path, sheet_name=sheet, header=None, dtype=object)
    for i in range(min(max_rows, len(raw))):
        if any(marker in _low(v) for v in raw.iloc[i] if pd.notna(v)):
            df = raw.iloc[i + 1:].copy()
            df.columns = [_low(v) if pd.notna(v) else 'col_%s' % j for j, v in enumerate(raw.iloc[i])]
            return df.dropna(how='all').reset_index(drop=True)
    raise ValueError('Не нашёл строку заголовков с "%s"' % marker)

def status_group(v):
    s = '' if v is None or pd.isna(v) else str(v).lower()
    if 'на закрыт' in s:
        return 'На закрытии'
    if 'закрыт' in s:
        return 'Закрыт'
    if 'работ' in s:
        return 'Работает'
    return s or '(пусто)'

sn_raw = read_with_header(SNIGUR_FILE, SNIGUR_SHEET)
sn = pd.DataFrame({
    'agr_key': sn_raw['номер договора'].map(norm_agr),
    'Номер договора (Снигур)': sn_raw['номер договора'].astype('string').str.strip(),
    'Статус (Снигур)': sn_raw['статус'].map(status_group),
    'Дата начала': to_date(sn_raw['дата начала действия договора']),
    'Дата окончания': to_date(sn_raw['дата окончания действия договора']),
    'Дата закрытия': to_date(sn_raw['дата закрытия']),
})
sn = sn[sn['agr_key'].notna()].drop_duplicates('agr_key').reset_index(drop=True)
print('Договоров у Снигура:', len(sn))
display(sn['Статус (Снигур)'].value_counts())

agr_parts = [fetch(
    "SELECT upper(regexp_replace(a.c_agr_number, '[[:space:]]', '')) AS agr_key, a.* "
    "FROM ods_alpha.scd1_agreements a "
    "WHERE upper(regexp_replace(a.c_agr_number, '[[:space:]]', '')) IN (%s)" % sql_in(part))
    for part in chunks(sn['agr_key'])]
agr = pd.concat(agr_parts, ignore_index=True)
agr = agr.loc[:, ~agr.columns.duplicated()]
agr['_deleted'] = agr.get('ods_deleted_flg', pd.Series('0', index=agr.index)).astype(str).eq('1')
agr['_sa'] = agr['acq_class'].astype(str).str.strip().str.upper().eq('SA')
agr['_from'] = to_date(agr['d_valid_from'])
agr = (agr.sort_values(['agr_key', '_deleted', '_sa', '_from'], ascending=[True, True, False, False])
       .drop_duplicates('agr_key'))
print('Нашлись в scd1_agreements по номеру:', len(agr), 'из', len(sn))

def id_str(s):
    return s.astype('string').str.strip().str.replace(r'\.0+$', '', regex=True)

agr['_id'] = id_str(agr['abs_agr_id'])
agr['_n_agr'] = id_str(agr['n_agr'])

cft_parts = [fetch("SELECT m.* FROM ods.scd1_z_r2_ip_merchants m WHERE cast(m.id AS string) IN (%s)" % sql_in(part))
             for part in chunks(agr['_id'].dropna())]
cft = pd.concat(cft_parts, ignore_index=True) if cft_parts else pd.DataFrame(columns=['id'])
cft['id'] = id_str(cft['id'])
cft = cft.drop_duplicates('id')
print('Нашлись в z_r2_ip_merchants по abs_agr_id:', len(cft))

terms_parts = [fetch('''
SELECT t.n_agr,
  max(CASE WHEN upper(trim(t.cf_ter_type)) = 'P' THEN t.d_valid_to END) AS p_last_to,
  sum(CASE WHEN upper(trim(t.cf_ter_type)) = 'P' AND t.d_valid_to IS NULL THEN 1 ELSE 0 END) AS p_open,
  min(CASE WHEN upper(trim(t.cf_ter_type)) = 'P' THEN t.d_valid_from END) AS p_first_from,
  max(t.d_valid_to) AS any_last_to,
  sum(CASE WHEN t.d_valid_to IS NULL THEN 1 ELSE 0 END) AS any_open
FROM ods_alpha.scd1_agr_terms t
WHERE coalesce(t.ods_deleted_flg, '0') <> '1' AND cast(t.n_agr AS string) IN (%s)
GROUP BY t.n_agr''' % sql_in(part)) for part in chunks(agr['_n_agr'].dropna())]
terms = pd.concat(terms_parts, ignore_index=True) if terms_parts else pd.DataFrame(
    columns=['n_agr', 'p_last_to', 'p_open', 'p_first_from', 'any_last_to', 'any_open'])
terms['n_agr'] = id_str(terms['n_agr'])
terms['p_last_to'] = terms['p_last_to'].where(pd.to_numeric(terms['p_open'], errors='coerce').fillna(0) == 0)
terms['any_last_to'] = terms['any_last_to'].where(pd.to_numeric(terms['any_open'], errors='coerce').fillna(0) == 0)
terms = terms[['n_agr', 'p_last_to', 'p_first_from', 'any_last_to']]

agr_w = agr.drop(columns=['_deleted', '_sa', '_from', '_id', '_n_agr']).add_prefix('agr.').rename(columns={'agr.agr_key': 'agr_key'})
agr_w['_id'] = agr['_id'].values
agr_w['_n_agr'] = agr['_n_agr'].values
wide = (sn.merge(agr_w, on='agr_key', how='left')
        .merge(cft.add_prefix('cft.'), left_on='_id', right_on='cft.id', how='left')
        .merge(terms.add_prefix('terms.'), left_on='_n_agr', right_on='terms.n_agr', how='left'))
found = wide['agr.n_agr'].notna()
print('В сравнении (есть в scd1_agreements):', int(found.sum()), '| с мерчантом ЦФТ:', int(wide['cft.id'].notna().sum()))


## 3. Какая колонка озера совпадает с датами Снигура

Для каждой даты Снигура и каждой колонки-даты озера:
- «оба заполнены» - есть дата и у Снигура, и в озере;
- «% точно» - доля точных совпадений среди «оба заполнены»;
- «% ±1 день» - разница не больше дня (часто дата закрытия = окончание + 1 или −1);
- «% пусто там же» - доля договоров без даты у Снигура, где и в озере пусто. Для даты закрытия это важно: открытые договоры не должны получать дату.


In [ ]:
date_like = set(date_desc['alias'] + '.' + date_desc['Колонка'])
date_like |= {'terms.p_last_to', 'terms.p_first_from', 'terms.any_last_to'}
cand = {}
for c in wide.columns:
    if c in date_like and found.any():
        d = to_date(wide.loc[found, c])
        raw_filled = wide.loc[found, c].notna() & ~wide.loc[found, c].astype(str).str.strip().str.upper().isin(EMPTY)
        if raw_filled.sum() == 0 or d.notna().sum() / raw_filled.sum() >= 0.8:
            cand[c] = d
print('Колонок-дат в сравнении:', len(cand))

def score(sn_dates, d, mask=None):
    if mask is not None:
        sn_dates, d = sn_dates[mask], d[mask]
    both = sn_dates.notna() & d.notna()
    nb_ = int(both.sum())
    eq = int(((sn_dates == d) & both).sum())
    near = int((((sn_dates - d).abs() <= pd.Timedelta(days=1)) & both).sum())
    sn_empty = sn_dates.isna()
    return {
        'Заполнено у Снигура': int(sn_dates.notna().sum()),
        'Заполнено в озере': int(d.notna().sum()),
        'Оба заполнены': nb_,
        'Совпадает точно': eq,
        '% точно': round(eq / nb_ * 100, 1) if nb_ else None,
        '% ±1 день': round(near / nb_ * 100, 1) if nb_ else None,
        '% пусто там же': round((sn_empty & d.isna()).sum() / sn_empty.sum() * 100, 1) if sn_empty.sum() else None,
    }

rank_parts = []
for target in ('Дата начала', 'Дата окончания', 'Дата закрытия'):
    s = wide.loc[found, target]
    rows = [dict(**{'Дата Снигура': target, 'Колонка озера': c}, **score(s, d)) for c, d in cand.items()]
    r = pd.DataFrame(rows)
    r = r[r['Совпадает точно'] > 0].sort_values(['Совпадает точно', '% пусто там же'], ascending=False)
    rank_parts.append(r)
    print(target)
    display(r.head(10))
ranking = pd.concat(rank_parts, ignore_index=True)

def best(target, manual):
    if manual:
        return manual
    r = ranking[ranking['Дата Снигура'] == target]
    return r.iloc[0]['Колонка озера'] if len(r) else None

start_col = best('Дата начала', START_COL)
end_col = best('Дата окончания', END_COL)
close_col = best('Дата закрытия', CLOSE_COL)
print('Начало ->', start_col, '| окончание ->', end_col, '| закрытие ->', close_col)

by_status = []
st = wide.loc[found, 'Статус (Снигур)']
for target, col in (('Дата закрытия', close_col), ('Дата окончания', end_col), ('Дата закрытия', 'agr.d_valid_to')):
    if not col or col not in cand:
        continue
    for g in st.unique():
        by_status.append(dict(**{'Дата Снигура': target, 'Колонка озера': col, 'Статус (Снигур)': g},
                              **score(wide.loc[found, target], cand[col], st == g)))
by_status = pd.DataFrame(by_status)
display(by_status)


## 4. «Действует на дату» по озеру и у Снигура

Правило по озеру: начало (`start_col`, иначе `d_valid_from`) не позже даты сверки, а дата закрытия (`close_col`) пустая или не раньше. Сравнивается с тремя вариантами «действует» у Снигура:
- `end` - более ранняя из окончания и закрытия (как сейчас по умолчанию в сверке);
- `close` - дата закрытия, а если пусто - дата окончания;
- `status` - статус не «Закрыт».

И для контроля - текущее правило моей сборки по `d_valid_to`.


In [ ]:
w = wide[found].copy()
lake_start = cand.get(start_col, to_date(w['agr.d_valid_from']))
started = lake_start.isna() | (lake_start <= as_of)

def open_on(d):
    return d.isna() | (d >= as_of)

lake_rules = {'Моя сборка (d_valid_to)': started & open_on(to_date(w['agr.d_valid_to']))}
if close_col in cand:
    lake_rules['Озеро, закрытие = %s' % close_col] = started & open_on(cand[close_col])
    lake_rules['Озеро, d_valid_to и %s' % close_col] = started & open_on(to_date(w['agr.d_valid_to'])) & open_on(cand[close_col])

sn_start_ok = w['Дата начала'].isna() | (w['Дата начала'] <= as_of)
sn_rules = {
    'end': sn_start_ok & open_on(w[['Дата окончания', 'Дата закрытия']].min(axis=1)),
    'close': sn_start_ok & open_on(w['Дата закрытия'].fillna(w['Дата окончания'])),
    'status': w['Статус (Снигур)'].ne('Закрыт'),
}

sim = []
for ln, lv in lake_rules.items():
    for sname, sv in sn_rules.items():
        sim.append({
            'Правило озера': ln, 'Правило Снигура': sname,
            'Действуют по озеру': int(lv.sum()), 'Действуют у Снигура': int(sv.sum()),
            'Совпало (действуют в обоих)': int((lv & sv).sum()),
            'Только по озеру': int((lv & ~sv).sum()), 'Только у Снигура': int((~lv & sv).sum()),
            '% совпадения статуса': round((lv == sv).mean() * 100, 1),
        })
sim = pd.DataFrame(sim)
display(sim)

show = ['Номер договора (Снигур)', 'Статус (Снигур)', 'Дата начала', 'Дата окончания', 'Дата закрытия',
        'agr.d_valid_from', 'agr.d_valid_to'] + [c for c in (start_col, end_col, close_col) if c and c in w.columns]
show = list(dict.fromkeys(show))
mismatch = pd.DataFrame()
if close_col in cand:
    lv = lake_rules['Озеро, закрытие = %s' % close_col]
    mismatch = w.loc[lv != sn_rules['close'], show].assign(
        **{'Действует по озеру': lv[lv != sn_rules['close']], 'Действует у Снигура (close)': sn_rules['close'][lv != sn_rules['close']]})
    display(mismatch.head(20))


## 5. Выгрузка

In [ ]:
def as_dates(df):
    df = df.copy()
    for c in df.columns:
        if pd.api.types.is_datetime64_any_dtype(df[c]):
            df[c] = df[c].dt.date
    return df

picked = pd.DataFrame([('Дата начала', start_col), ('Дата окончания', end_col), ('Дата закрытия', close_col)],
                      columns=['Дата Снигура', 'Лучшая колонка озера'])
with pd.ExcelWriter(OUT_FILE, engine='openpyxl') as writer:
    row = 0
    for block in (picked, sim):
        block.to_excel(writer, index=False, sheet_name='Сводка', startrow=row)
        row += len(block) + 2
    ranking.to_excel(writer, index=False, sheet_name='Совпадение дат')
    by_status.to_excel(writer, index=False, sheet_name='По статусам')
    date_desc.to_excel(writer, index=False, sheet_name='Колонки-даты')
    desc.to_excel(writer, index=False, sheet_name='Все колонки')
    if len(mismatch):
        as_dates(mismatch).to_excel(writer, index=False, sheet_name='Расходится статус')
print(OUT_FILE)
